## 1. Setup & Imports

In [4]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from ucimlrepo import fetch_ucirepo
import warnings

warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)

RANDOM_STATE = 42
OUTPUT_DIR = os.path.join('..', '..', 'data', 'spambase_50_50')
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA Version: {torch.version.cuda}')

Using device: cuda
GPU: NVIDIA RTX A6000
CUDA Version: 11.8


## 2. Load & Preprocess Spambase

In [5]:
spambase = fetch_ucirepo(id=94)
X_raw = spambase.data.features
y_raw = spambase.data.targets

df = pd.concat([X_raw, y_raw], axis=1)
target_col = y_raw.columns[0]
feature_names = X_raw.columns.tolist()

print(f'Dataset: {spambase.metadata["name"]}')
print(f'Samples: {df.shape[0]}, Features: {len(feature_names)}')
print(f'Class distribution:\n{df[target_col].value_counts().to_string()}')

Dataset: Spambase
Samples: 4601, Features: 57
Class distribution:
Class
0    2788
1    1813


In [6]:
scaler = MinMaxScaler()
df[feature_names] = scaler.fit_transform(df[feature_names])

weights = abs(df.corr()[target_col]).drop(target_col).values
weights = weights / (np.linalg.norm(weights) + 1e-8)

bounds = [df[feature_names].min().values, df[feature_names].max().values]

print(f'Scaling: MinMaxScaler to [0, 1]')
print(f'Feature weights shape: {weights.shape}')
print(f'Bounds shape: {[b.shape for b in bounds]}')

Scaling: MinMaxScaler to [0, 1]
Feature weights shape: (57,)
Bounds shape: [(57,), (57,)]


## 3. 50/50 Train-Test Split

In [7]:
X_all = df[feature_names].values
y_all = df[target_col].values.astype(int)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_all, y_all, test_size=0.5, random_state=RANDOM_STATE, stratify=y_all
)

print(f'Train set: {X_train_np.shape[0]} samples')
print(f'  Class 0: {(y_train_np == 0).sum()}, Class 1: {(y_train_np == 1).sum()}')
print(f'Test set:  {X_test_np.shape[0]} samples')
print(f'  Class 0: {(y_test_np == 0).sum()}, Class 1: {(y_test_np == 1).sum()}')

Train set: 2300 samples
  Class 0: 1394, Class 1: 906
Test set:  2301 samples
  Class 0: 1394, Class 1: 907


In [8]:
np.savez(
    os.path.join(OUTPUT_DIR, 'train_test_data.npz'),
    X_train=X_train_np, y_train=y_train_np,
    X_test=X_test_np, y_test=y_test_np,
)

metadata = {
    'dataset': 'Spambase (UCI ID 94)',
    'split_ratio': '50/50',
    'random_state': RANDOM_STATE,
    'stratified': True,
    'n_features': len(feature_names),
    'feature_names': feature_names,
    'n_train': int(X_train_np.shape[0]),
    'n_test': int(X_test_np.shape[0]),
    'scaler': 'MinMaxScaler [0, 1]',
}
with open(os.path.join(OUTPUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

np.savez(
    os.path.join(OUTPUT_DIR, 'preprocessing.npz'),
    weights=weights,
    bounds_min=bounds[0],
    bounds_max=bounds[1],
)

print(f'Saved train/test data, metadata, and preprocessing artifacts to {OUTPUT_DIR}/')

Saved train/test data, metadata, and preprocessing artifacts to spambase_50_50/


## 4. Define & Train MLP Classifier

In [9]:
class SpambaseNet(nn.Module):
    def __init__(self, D_in):
        super(SpambaseNet, self).__init__()
        self.layer = nn.Sequential(
            nn.Linear(D_in, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 2), nn.Softmax(dim=-1)
        )

    def forward(self, x):
        return self.layer(x)

D_in = len(feature_names)
model = SpambaseNet(D_in).to(device)
print(model)

SpambaseNet(
  (layer): Sequential(
    (0): Linear(in_features=57, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=2, bias=True)
    (5): Softmax(dim=-1)
  )
)


In [10]:
X_train_t = torch.FloatTensor(X_train_np).to(device)
y_train_t = torch.nn.functional.one_hot(
    torch.LongTensor(y_train_np), num_classes=2
).float().to(device)

X_test_t = torch.FloatTensor(X_test_np).to(device)
y_test_t = torch.nn.functional.one_hot(
    torch.LongTensor(y_test_np), num_classes=2
).float().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()
N_EPOCHS = 100

print('Training...')
model.train()
for epoch in range(N_EPOCHS):
    optimizer.zero_grad()
    output = model(X_train_t)
    loss = criterion(output, y_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 20 == 0:
        print(f'  Epoch {epoch+1:3d}/{N_EPOCHS} — Loss: {loss.item():.4f}')

model.eval()
print('Training complete.')

Training...
  Epoch  20/100 — Loss: 0.6562
  Epoch  40/100 — Loss: 0.5973
  Epoch  60/100 — Loss: 0.4696
  Epoch  80/100 — Loss: 0.3404
  Epoch 100/100 — Loss: 0.2751
Training complete.


## 5. Evaluate on Train & Test Sets

In [11]:
with torch.no_grad():
    train_preds = model(X_train_t).argmax(dim=1).cpu().numpy()
    test_preds = model(X_test_t).argmax(dim=1).cpu().numpy()

train_acc = (train_preds == y_train_np).mean()
test_acc = (test_preds == y_test_np).mean()

print(f'Train Accuracy: {train_acc:.2%}')
print(f'Test Accuracy:  {test_acc:.2%}')
print()
print('Test Set Classification Report:')
print(classification_report(y_test_np, test_preds, target_names=['Not Spam', 'Spam']))
print('Confusion Matrix:')
print(confusion_matrix(y_test_np, test_preds))

Train Accuracy: 91.09%
Test Accuracy:  89.96%

Test Set Classification Report:
              precision    recall  f1-score   support

    Not Spam       0.90      0.94      0.92      1394
        Spam       0.90      0.84      0.87       907

    accuracy                           0.90      2301
   macro avg       0.90      0.89      0.89      2301
weighted avg       0.90      0.90      0.90      2301

Confusion Matrix:
[[1306   88]
 [ 143  764]]


## 6. Save Trained Model

In [12]:
model_path = os.path.join(OUTPUT_DIR, 'spambase_mlp.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'SpambaseNet',
    'D_in': D_in,
    'hidden_layers': [64, 32],
    'n_classes': 2,
    'train_accuracy': float(train_acc),
    'test_accuracy': float(test_acc),
    'optimizer': 'Adam',
    'lr': 0.001,
    'epochs': N_EPOCHS,
    'loss': 'BCELoss',
}, model_path)

print(f'Model saved to {model_path}')
print(f'\nTo reload later:')
print(f'  checkpoint = torch.load("{model_path}")')
print(f'  model = SpambaseNet(checkpoint["D_in"]).to(device)')
print(f'  model.load_state_dict(checkpoint["model_state_dict"])')
print(f'  model.eval()')

Model saved to spambase_50_50/spambase_mlp.pth

To reload later:
  checkpoint = torch.load("spambase_50_50/spambase_mlp.pth")
  model = SpambaseNet(checkpoint["D_in"]).to(device)
  model.load_state_dict(checkpoint["model_state_dict"])
  model.eval()


## 7. Summary

All artifacts saved under `experiments/data/spambase_50_50/`:

| File | Contents |
|------|----------|
| `train_test_data.npz` | `X_train`, `y_train`, `X_test`, `y_test` (numpy arrays, scaled) |
| `preprocessing.npz` | `weights` (L2-normalized correlation weights), `bounds_min`, `bounds_max` |
| `metadata.json` | Split configuration, feature names, sample counts |
| `spambase_mlp.pth` | Trained SpambaseNet state dict + hyperparameters |